In [ ]:
# ============================================================
# EXPERIMENT 6A: NAIVE BAYES CLASSIFICATION FROM SCRATCH
# Student Admission Prediction (Categorical + Laplace Smoothing)
# ============================================================
import numpy as np
import pandas as pd

# Step 1 & 2: Create Dataset and Separate Features/Target
df = pd.DataFrame({
    "Study_Hours": ["Low", "Low", "Medium", "Medium", "High", "High", "High", "Medium", "Low", "High"],
    "Attendance": ["Low", "High", "Medium", "High", "Medium", "High", "High", "Medium", "Low", "Medium"],
    "Previous_Marks": ["Low", "Medium", "Medium", "High", "High", "High", "Medium", "High", "Low", "High"],
    "Admission": ["No", "No", "No", "Yes", "Yes", "Yes", "Yes", "Yes", "No", "Yes"]
})
features = ["Study_Hours", "Attendance", "Previous_Marks"]
target = "Admission"
classes = df[target].unique()
print(f"Student Admission Dataset:\n{df}")

# Step 3: Calculate Prior Probabilities
priors = {cls: (df[target] == cls).sum() / len(df) for cls in classes}
print("\nPrior Probabilities:\n--------------------------------")
for cls, probability in priors.items():
    print(f"{cls} : {round(probability, 4)}")

# Step 4: Conditional Probability Using Laplace Smoothing
def conditional_probability(feature, value, cls):
    class_data = df[df[target] == cls]
    return (len(class_data[class_data[feature] == value]) + 1) / (len(class_data) + df[feature].nunique())

# Step 5: Display Conditional Probabilities
print("\nConditional Probabilities:\n================================")
for cls in classes:
    print(f"\nClass: {cls}")
    for feature in features:
        for value in df[feature].unique():
            print(f"P({feature}={value} | Admission={cls}) = {conditional_probability(feature, value, cls):.4f}")

# Step 6: Naive Bayes Prediction Function (Prior x Product of Likelihoods)
def predict_naive_bayes(sample):
    probabilities = {}
    for cls in classes:
        probability = priors[cls]
        for feature in features:
            probability *= conditional_probability(feature, sample[feature], cls)
        probabilities[cls] = probability
    return max(probabilities, key=probabilities.get), probabilities

# Step 7 & 8: Predict New Student and Display Class Probabilities
new_student = {"Study_Hours": "High", "Attendance": "High", "Previous_Marks": "High"}
prediction, probabilities = predict_naive_bayes(new_student)
print("\n===================================\nNEW STUDENT\n===================================")
print(f"Study Hours : {new_student['Study_Hours']}\nAttendance : {new_student['Attendance']}\nPrevious Marks : {new_student['Previous_Marks']}")

print("\nClass Probabilities:")
for cls, probability in probabilities.items():
    print(f"{cls} : {round(probability, 6)}")

# Step 9 & 10: Final Prediction and Normalized Probabilities
print("\n===================================\nNAIVE BAYES PREDICTION\n===================================")
print(f"Predicted Admission: {prediction}")

total_probability = sum(probabilities.values())
print("\nNormalized Probabilities:")
for cls, probability in probabilities.items():
    print(f"{cls} : {round(probability / total_probability, 4)}")

# Step 11 & 12: Multiple Predictions and Display Results
test_students = pd.DataFrame({
    "Study_Hours": ["High", "Low", "Medium", "High"],
    "Attendance": ["High", "Low", "Medium", "Medium"],
    "Previous_Marks": ["High", "Low", "Medium", "High"]
})
test_students["Predicted_Admission"] = [predict_naive_bayes(r)[0] for r in test_students.to_dict("records")]

print("\n===================================\nMULTIPLE STUDENT PREDICTIONS\n===================================")
print(test_students)

In [ ]:
# ============================================================
# EXPERIMENT 6B: GAUSSIAN NAIVE BAYES USING SCIKIT-LEARN
# Real-World Dataset: Iris
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Step 1-3: Load Dataset, Create DataFrame, and Display Info
iris = load_iris()
X, y = iris.data, iris.target
feature_names, target_names = iris.feature_names, iris.target_names

df = pd.DataFrame(X, columns=feature_names)
df["Target"] = y
df["Species"] = df["Target"].map({0: "setosa", 1: "versicolor", 2: "virginica"})
print(f"Dataset Shape:\n{df.shape}\n\nFirst Five Records:\n{df.head()}")
print(f"\nFeature Names:\n{feature_names}\n\nTarget Classes:\n{target_names}\n\nClass Distribution:\n{df['Species'].value_counts()}")

# Step 4-5: Train-Test Split (Stratified)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print(f"\nTraining Samples: {X_train.shape[0]}\nTesting Samples: {X_test.shape[0]}")

# Step 6-9: Create, Train, Predict, and Compare
model = GaussianNB().fit(X_train, y_train)
y_pred = model.predict(X_test)

results = pd.DataFrame({"Actual": y_test, "Predicted": y_pred})
results["Actual Species"] = results["Actual"].map(dict(enumerate(target_names)))
results["Predicted Species"] = results["Predicted"].map(dict(enumerate(target_names)))
print(f"\nActual vs Predicted:\n{results}")

# Step 10: Accuracy
accuracy = accuracy_score(y_test, y_pred)
print("\n===================================\nGAUSSIAN NAIVE BAYES\n===================================")
print(f"Accuracy: {round(accuracy, 4)}\nAccuracy (%): {round(accuracy * 100, 2)}")

# Steps 11-12: Confusion Matrix and Classification Report
cm = confusion_matrix(y_test, y_pred)
print(f"\nConfusion Matrix:\n{cm}")
print(f"\nClassification Report:\n{classification_report(y_test, y_pred, target_names=target_names)}")

# Step 13: Class Probabilities
probability_results = pd.DataFrame(model.predict_proba(X_test), columns=target_names)
print(f"\nClass Probabilities:\n{probability_results.head(10)}")

# Step 14: Predict New Flower
new_flower = np.array([[5.1, 3.5, 1.4, 0.2]])
new_prediction, new_probability = model.predict(new_flower), model.predict_proba(new_flower)
print("\n===================================\nNEW FLOWER PREDICTION\n===================================")
print(f"Predicted Class: {target_names[new_prediction[0]]}\n\nClass Probabilities:")
for class_name, probability in zip(target_names, new_probability[0]):
    print(f"{class_name} : {round(probability, 4)}")

# Step 15: Confusion Matrix Visualization
plt.figure(figsize=(7, 5))
plt.imshow(cm, interpolation="nearest"), plt.title("Gaussian Naive Bayes Confusion Matrix")
plt.xlabel("Predicted Class"), plt.ylabel("Actual Class")
plt.xticks(range(len(target_names)), target_names), plt.yticks(range(len(target_names)), target_names)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")
plt.colorbar(), plt.tight_layout(), plt.show()

# Step 16: Actual vs Predicted Plot
sample_number = np.arange(len(y_test))
plt.figure(figsize=(9, 5))
plt.scatter(sample_number, y_test, label="Actual", marker="o")
plt.scatter(sample_number, y_pred, label="Predicted", marker="x")
plt.xlabel("Test Sample Number"), plt.ylabel("Class"), plt.title("Naive Bayes: Actual vs Predicted")
plt.yticks(range(len(target_names)), target_names), plt.legend(), plt.grid(True), plt.tight_layout(), plt.show()